In [1]:
import pathlib
import cogent3
import madb 
from typing import Dict, List
import dataclasses
import shutil

comparison_species = 'pan_troglodytes'
comparison_species_common = 'chimp'
pair_name = 'human_'+comparison_species_common
thesis_data = pathlib.Path('~/source/ensembl/thesis_data')
source_alignment_path = thesis_data/'ensembl_alignments'/pair_name
output_path_root = thesis_data/'recs'/pair_name 
limit = 20

# Extract pair of primates from alignments of all primates
 - rename sequences to species name 
 - select only the pair of species
 - remove gaps

In [2]:
@cogent3.app.composable.define_app
def rename(align: cogent3.app.typing.AlignedSeqsType)->cogent3.app.typing.AlignedSeqsType:
    return align.rename_seqs(lambda x: x.split(':')[0])

loader = cogent3.get_app('load_aligned', moltype='dna')
renamer = rename()
select_seqs = cogent3.get_app('take_named_seqs','homo_sapiens',comparison_species)
omit_gaps = cogent3.get_app('omit_gap_pos', moltype="dna")
in_dstore = cogent3.open_data_store('~/source/ensembl/primates100', suffix='fa') 
out_dstore = cogent3.open_data_store(source_alignment_path, suffix='fa', mode='w') 
writer = cogent3.get_app('write_seqs', data_store = out_dstore)

app = loader + renamer + select_seqs + omit_gaps + writer 
app.apply_to(in_dstore, show_progress=True, parallel=True)
out_dstore.describe

   0%|          |00:00<?

completed,120
not_completed,1
logs,1


# Create dataclass for recoding results as JSON

In [3]:
import time
from typing import Any


@dataclasses.dataclass
class thesis_rec:
    unique_id : str
    unaligned_seqs : Dict[str,str]
    ensembl_alignment : Dict[str,str]
    ensembl_pd : float
    cogent3_alignment : Dict[str,str]
    cogent3_time : float
    cogent3_pd : float
    jaccard_distance : Dict[int, float]
    madb_alignment : Dict[int,Dict[str,str]]
    madb_time : Dict[int,float]
    madb_score : Dict[int, float]    
    madb_distance : Dict[int, float]
    madb_bubbles : Dict[int,int]
    madb_braids : Dict[int, int]
    madb_cycles : Dict[int, bool]
    madb_longest_braid : Dict[int, str]
    madb_ungapped_smith_waterman : str
    def to_rich_dict(self):
        return {
            'unique_id' : self.unique_id,
            'unaligned_seqs' : self.unaligned_seqs,
            'ensembl_alignment' : self.ensembl_alignment,
            'ensembl_pd' : self.ensembl_pd,
            'cogent3_alignment' : self.cogent3_alignment,
            'cogent3_time' : self.cogent3_time,
            'cogent3_pd' : self.cogent3_pd,
            'jaccard_distance' : self.jaccard_distance,
            'madb_alignment' : self.madb_alignment,
            'madb_time' : self.madb_time,
            'madb_score' : self.madb_score,
            'madb_distance' : self.madb_distance,
            'madb_bubbles' : self.madb_bubbles,
            'madb_braids' : self.madb_braids,
            'madb_cycles' : self.madb_cycles,
            'madb_longest_braid' : self.madb_longest_braid,
            'madb_ungapped_smith_waterman' : self.madb_ungapped_smith_waterman
        }
    @classmethod
    def from_rich_dict(cls, d: Dict[str, Any]) -> "thesis_rec":
        return cls(
            unique_id = d['unique_id'],
            unaligned_seqs = d['unaligned_seqs'],
            ensembl_alignment = d['ensembl_alignment'],
            ensembl_pd = d['ensembl_pd'],
            cogent3_alignment = d['cogent3_alignment'],
            cogent3_time = d['cogent3_time'],
            cogent3_pd = d['cogent3_pd'],
            jaccard_distance = d['jaccard_distance'],
            madb_alignment = d['madb_alignment'],
            madb_time = d['madb_time'],
            madb_score = d['madb_score'],
            madb_distance = d['madb_distance'],
            madb_bubbles = d['madb_bubbles'],
            madb_braids = d['madb_braids'],
            madb_cycles = d['madb_cycles'],
            madb_longest_braid = d['madb_longest_braid'],
            madb_ungapped_smith_waterman = d['madb_ungapped_smith_waterman']
        )
    def align_cogent3(self) -> "thesis_rec":
        # Perform pairwise alignment using Cogent3
        unaligned = cogent3.make_unaligned_seqs(data = self.unaligned_seqs, moltype="dna")
        # Perform pairwise alignment
        for kmer_size in range(10, 100, 5):
            jaccard = cogent3.get_app('jaccard_dist', k=kmer_size)
            jcdists = jaccard(unaligned)
            self.jaccard_distance[kmer_size] = jcdists[unaligned.names[0], unaligned.names[1]]
        start_time = time.time()
        # Perform global pairwise alignment
        score_matrix = cogent3.align.align.make_dna_scoring_dict(match=5, transition=-2, transversion=-4)
        gap_penalty = 4
        gap_extend = 1 
        alignment = cogent3.align.global_pairwise(unaligned.seqs[0], unaligned.seqs[1], score_matrix, gap_penalty, gap_extend)
        self.cogent3_alignment = alignment.to_dict()
        self.cogent3_time = time.time() - start_time
        self.cogent3_pd = alignment.distance_matrix('pdist')[alignment.names[0], alignment.names[1]]
        return self
    
    def align_madb(self) -> "thesis_rec":
        for kmer_size in range(10, 100, 5):
            # Perform pairwise alignment using MADB
            start_time = time.time()
            graph = madb.make_graph(self.unaligned_seqs, kmer_size=10, moltype=cogent3.DNA)
            self.madb_time[kmer_size] = time.time() - start_time
            self.madb_alignment[kmer_size] = graph.to_dict()
            braid_diff, braid_nucleotides, total_braids, total_bubbles, has_cycles   = graph.distance(braid_bubble_counts=True)
            self.madb_distance[kmer_size] = braid_diff / braid_nucleotides
            self.madb_bubbles[kmer_size] = total_bubbles
            self.madb_braids[kmer_size] = total_braids
            self.madb_cycles[kmer_size] = has_cycles
            self.madb_longest_braid[kmer_size] = graph.longest_braid()
            seqs = list(self.unaligned_seqs.values())
            self.madb_ungapped_smith_waterman = madb.smith_waterman_ungapped(seqs[0], seqs[1])
        return self

@cogent3.app.composable.define_app
def create_thesis_rec(align: cogent3.app.typing.AlignedSeqsType)-> cogent3.app.typing.SerialisableType:
    ensembl_pd = align.take_seqs(align.names)
    ensembl_alignment = align.to_dict()
    unaligned = align.degap()
    distance_mat = align.distance_matrix('pdist')
    ensembl_pdist = distance_mat[align.names[0],align.names[1]]
    new_rec = thesis_rec(
        unique_id = cogent3.app.data_store.get_data_source(align),
        unaligned_seqs = unaligned.to_dict(),
        ensembl_alignment = ensembl_alignment,
        ensembl_pd = ensembl_pdist,
        cogent3_alignment = {},
        cogent3_time = 0.0,
        cogent3_pd = 0.0,
        jaccard_distance = {},
        madb_alignment = {},
        madb_time = {},
        madb_score = {},
        madb_distance = {},
        madb_bubbles = {},
        madb_braids = {},
        madb_cycles = {},
        madb_longest_braid = {},
        madb_ungapped_smith_waterman = {}
    )
    return new_rec



# degap and use cogent3 to re-align the sequences
 - record the alignment
 - how long it took 
 - the pdist of the alignment

In [4]:
in_dstore = cogent3.open_data_store(source_alignment_path, suffix='fa', limit=limit, mode='r') 
out_dstore = cogent3.open_data_store(output_path_root/'0', suffix='json', mode='w') 
loader = cogent3.get_app('load_aligned', moltype='dna')
writer = cogent3.get_app('write_json', data_store = out_dstore)
select_seqs = cogent3.get_app('take_named_seqs','homo_sapiens',comparison_species)
omit_gaps = cogent3.get_app('omit_gap_pos', moltype="dna")

app = loader + select_seqs + omit_gaps + create_thesis_rec() + writer 
app.apply_to(in_dstore, show_progress=True, parallel=True)
out_dstore.describe

   0%|          |00:00<?

completed,20
not_completed,0
logs,1


In [5]:
out_dstore.summary_not_completed

0 rows x 5 columns
unset columns: 'type', 'origin', 'message', 'num', 'source'

# Align using cogent3

In [6]:
@cogent3.app.composable.define_app
def thesis_rec_cogent3_align(rec : dict) -> cogent3.app.typing.SerialisableType: 
    rec = thesis_rec.from_rich_dict(rec)
    rec.align_cogent3()
    return rec

in_dstore = cogent3.open_data_store(output_path_root/'0', suffix='json', mode='r') 
out_dstore = cogent3.open_data_store(output_path_root/'1', suffix='json', mode='w') 
loader = cogent3.get_app('load_json')
writer = cogent3.get_app('write_json', data_store = out_dstore)

app = loader + thesis_rec_cogent3_align() + writer 
app.apply_to(in_dstore, show_progress=True, parallel=True)
out_dstore.describe

   0%|          |00:00<?

completed,20
not_completed,0
logs,1


In [7]:
loader2 = cogent3.get_app('load_json')
rec = loader2(in_dstore[3])
thesis_rec.from_rich_dict(rec)

thesis_rec(unique_id='ENSG00000169717.fa', unaligned_seqs={'homo_sapiens': 'TTCCGGCATTGGTCCTTTATTGAACATCCTCCCAAAGCTGGGGCTAGGGTTCACCCCCCACGGTACCCAACGAGAAGCGGCCTTCAGAAGCATCTTCTCTGCACCACGGAGGTCCCAAACTCCTTGAAGTCTGCGGCGGTGACCCACATCTGCTTGAAGCTACTCAGAGAGGTGACGATGGAGGCTCCAATCCAGGTGGAGAACCACCGGTCGGGGGGAGCCGTGATCTTGATGGGGGTGTCCTTGGAGGCCAGCTGCTCCAGCTCCTTGAGAAGCCGGTCATCCAGCCCGTGGAACAGGGTAGTGCCCCCCGACAGCACAATCTCCCCAAAGAGGATCTTCTGGATGTCGGTATCACACTTGGTGATGCTGCTGGAGACCATATTCGAGAGCCCGGGGCTCTGGCTGCCCAGCTGCTGGGGCACGAACAGGGCCTCGGGCGCCTGGTGCAGCGGGTCCCCGAGGCTGATGATGTTCCCGTCGGGCAGCTTGTACTCCCTCAGGACCTCCTCCGGCCTCCGGGAAAGCTCCTTCTCGGGCTCCAAGGCCACGTAGCACAGCTTCTTTTTGATGTCGTCCACGAGACCCTTGTCCAGCTGGCAGGGGAAGGTGTGGCCGCTGGCCAGGAGCAGCTGCATGAGGAGCTCCGTGATGTCCCTGCCCGCCACGTGGAGCTTGGTGACTGCGTGGGGCAGGGAGTAACCCTCAAAGATGGGGACAGTGCAGGTGACCGCATCCCCGCTGTCCACCACCAGGCCCGTGACACAGGCAGAGGCGTAGAGAGCCAGCACCGCCTGGTCCGACAGGTAGAAAGCGGGCACGCCGAAGTTCTCGAACATGACTTCTGCCATCTTCTCACGGTTCTCCCTGGGGTTCAGGGAGGGCTCCGTTGCAAGCAGGGGCTGGTCGCTGGGTTTCACGCCT

In [8]:
if len(out_dstore.not_completed) > 0:
    print(cogent3.util.deserialise.deserialise_object(out_dstore.not_completed[0].read()))

In [9]:
out_dstore.summary_not_completed


0 rows x 5 columns
unset columns: 'type', 'origin', 'message', 'num', 'source'

In [10]:
@cogent3.app.composable.define_app
def thesis_rec_madb_align(d : dict) -> cogent3.app.typing.SerialisableType: 
    return thesis_rec.from_rich_dict(d).align_madb()

in_dstore = cogent3.open_data_store(output_path_root/'1', suffix='json', mode='r') 
out_dstore = cogent3.open_data_store(output_path_root/'2', suffix='json', mode='w') 
loader = cogent3.get_app('load_json')
writer = cogent3.get_app('write_json', data_store = out_dstore)

app = loader + thesis_rec_madb_align() + writer 
app.apply_to(in_dstore, show_progress=True, parallel=True)
out_dstore.describe

   0%|          |00:00<?

Exception ignored in: <generator object as_completed at 0x7331359649d0>
Traceback (most recent call last):
  File "/home/richard/source/ensembl/.venv/lib/python3.12/site-packages/cogent3/util/parallel.py", line 240, in as_completed
    yield from _as_completed_mproc(f, s, max_workers)
  File "/home/richard/source/ensembl/.venv/lib/python3.12/site-packages/cogent3/util/parallel.py", line 220, in _as_completed_mproc
    with loky.get_reusable_executor(max_workers=max_workers) as executor:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/richard/.local/share/uv/python/cpython-3.12.9-linux-x86_64-gnu/lib/python3.12/concurrent/futures/_base.py", line 647, in __exit__
    self.shutdown(wait=True)
  File "/home/richard/source/ensembl/.venv/lib/python3.12/site-packages/loky/process_executor.py", line 1333, in shutdown
    executor_manager_thread.join()
  File "/home/richard/.local/share/uv/python/cpython-3.12.9-linux-x86_64-gnu/lib/python3.12/threading.py", line 1149,

TypeError: Object of type DnaSequence is not JSON serializable

In [ ]:
out_dstore.summary_not_completed

In [ ]:
if len(out_dstore.not_completed) > 0:
    print(cogent3.util.deserialise.deserialise_object(out_dstore.not_completed[0].read()))